# learn-to-remember: M5 experiments 3-6 (one experiment per session)

Needs M3 (model choice) and M4 (experiment 2 adapter in `checkpoints/notes_l0.2`, for exp6) done first.
Set `EXP` and run all cells. Training runs resume: if a session ends, add this notebook's previous output as an input
and run again with the same `EXP`.

| EXP | what | spec |
|---|---|---|
| exp3 | train without notes (λ 0.2), eval no notes | RQ1 2x2 |
| exp4_l0 / exp4_l0.5 | train with notes at λ 0 / 0.5, eval v3 | RQ2 |
| exp5 | train on v0 notes, eval v0-v3 and company B | RQ3, RQ4 |
| exp6 | eval only: exp-2 adapter on company B and on v0-v2 | RQ3, RQ4 |
| drift_base | eval only: base model on v0-v2 | RQ3 |

In [ ]:
import os, subprocess, glob
REPO = "/kaggle/working/learn-to-remember"
if not os.path.exists(REPO):
    if os.path.exists("/kaggle/input/learn-to-remember"):
        subprocess.run(["cp", "-r", "/kaggle/input/learn-to-remember", REPO], check=True)
    else:
        subprocess.run(["git", "clone", "https://github.com/Vivek-Chaudhari30/learn-to-remember.git", REPO], check=True)
os.chdir(REPO)
!pip install -q "transformers>=4.51,<6" "peft>=0.12" "accelerate>=0.34" pyyaml

In [ ]:
EXP = "exp3"                               # exp3 | exp4_l0 | exp4_l0.5 | exp5 | exp6 | drift_base
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"       # <- the M3 choice
DTYPE, MICRO = "fp16", 2                   # T4: fp16. Lower MICRO to 1 on OOM
CKPT = "/kaggle/working/checkpoints"
# bring back checkpoints from earlier sessions / the M4 notebook output (added as inputs)
for st in glob.glob("/kaggle/input/*/checkpoints/*/training_state.pt"):
    run_dir = os.path.dirname(st); dst = os.path.join(CKPT, os.path.basename(run_dir))
    if not os.path.exists(os.path.join(dst, "training_state.pt")):
        os.makedirs(CKPT, exist_ok=True); subprocess.run(["cp", "-r", run_dir, dst], check=True); print("restored", dst)
for logdir in glob.glob("/kaggle/input/*/learn-to-remember/results/train/*"):
    dst = os.path.join("results/train", os.path.basename(logdir))
    if not os.path.exists(dst):
        subprocess.run(["cp", "-r", logdir, dst], check=True)

In [ ]:
!MODEL={MODEL} DTYPE={DTYPE} MICRO={MICRO} CKPT={CKPT} bash scripts/run_m5.sh {EXP} 2>&1 | grep -v -i warn

In [ ]:
!zip -qr /kaggle/working/m5_{EXP}_results.zip results/train results/trained results/baselines 2>/dev/null; ls -lh /kaggle/working/*.zip